In [1]:
import pandas as pd
import numpy as np
import psycopg
from psycopg import sql
import os
import sys
from dotenv import load_dotenv
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"

In [19]:
## Top hero winrates
columns=['id', 'didRadiantWin', 'gameVersionId', 'heroId', 'isRadiant', 'isVictory', 'displayName']
query = '''
    SELECT md.{0}, md.{1}, md.{2}, mp.{3}, mp.{4}, mp.{5}, hero_details.{6} 
    FROM match_details md
    INNER JOIN match_players mp
    ON md.id = mp.match_id
    INNER JOIN hero_details
    ON mp."heroId" = hero_details.id;
'''
df = dbf.query_select_to_df(conn_str, query, table_name='match_details', columns=columns, identifiers=columns)
hero_stats = df.groupby(['displayName'])['isVictory'].aggregate(['mean', 'count'])
hero_stats.columns = ['Win rate', 'Games played']
hero_stats['Win rate'] = (hero_stats['Win rate']*100).round(2)
hero_stats = hero_stats.sort_values(by='Win rate', ascending=False)
hero_stats = hero_stats[hero_stats['Games played'] >= 100]
hero_stats.head(10)

,Win rate,Games played
displayName,,
Keeper of the Light,60.36,169
Chen,59.84,254
Crystal Maiden,58.00,100
Elder Titan,57.78,135
Bristleback,57.21,402
Nature's Prophet,56.72,506
Naga Siren,56.54,283
Bane,56.10,246
Ember Spirit,56.06,503


In [22]:
## Pick and ban rates
## IMPORTANT: matches were dropped that had no pick-ban phase and where a pick/ban was missing
query = '''
    SELECT mpb.*, hero_details.name, hero_details."displayName"
    FROM match_pick_bans mpb
    INNER JOIN hero_details
    ON mpb."heroId" = hero_details.id;
'''
results = dbf.query_select(conn_str, query)
df = pd.DataFrame(results, columns=['id', 'matchId', 'isPick', 'heroId', 'order', 'isRadiant', 'heroName', 'displayName'])
orders = df.groupby('matchId')['order'].max().sort_values(ascending=True)
matches_to_drop = orders.loc[lambda x: x == 23].index
df = df[df['matchId'].isin(matches_to_drop)]

In [15]:
first_bans = df[df['order'] <= 6]
first_bans = first_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_ban_rates = first_bans / len(df['matchId'].unique())

first_picks = df[(df['order'] >= 7) & (df['order'] <= 8)] 
first_picks = first_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
first_pick_rates = first_picks / len(df['matchId'].unique())

second_bans = df[(df['order'] >= 9) & (df['order'] <= 11)]
second_bans = second_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_ban_rates = second_bans / len(df['matchId'].unique())

second_picks = df[(df['order'] >= 12) & (df['order'] <= 17)] 
second_picks = second_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
second_pick_rates = second_picks / len(df['matchId'].unique())

third_bans = df[(df['order'] >= 18) & (df['order'] <= 21)]
third_bans = third_bans.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_ban_rates = third_bans / len(df['matchId'].unique())

third_picks = df[(df['order'] >= 22)] 
third_picks = third_picks.groupby('displayName')['heroName'].count().sort_values(ascending=False)
third_pick_rates = third_picks / len(df['matchId'].unique())

overall_pbs = df.groupby('displayName')['displayName'].count().sort_values(ascending=False)
overal_pbs_rates = overall_pbs / len(df['matchId'].unique())

In [ ]:
df[df['isPick'] == True][['isPick', 'order']].value_counts() ## First pick is always 7

isPick  order
True    7        2984
        8        2984
        12       2984
        13       2984
        14       2984
        15       2984
        16       2984
        17       2984
        22       2984
        23       2984
Name: count, dtype: int64

In [35]:
query = '''
    SELECT id, match_details."didRadiantWin"
    FROM match_details
'''
df_match_ids = dbf.query_select_to_df(conn_str, query, 'match_details', columns=['match_id', 'didRadiantWin'])

In [42]:
df_match_ids[df_match_ids['match_id'] == row['matchId']]

,match_id,didRadiantWin
0,8183642521,True


In [75]:
fp_rad_wins = 0 #First pick wins for radiant
fp_rad_total = 0
fp_dire_wins = 0 #Second pick radiant wins
fp_dire_total = 0
for idx, row in df[df['order'] == 7].iterrows():
    if row['isRadiant']:
        if df_match_ids[df_match_ids['match_id'] == row['matchId']]['didRadiantWin'].values[0]:
            fp_rad_wins += 1
        fp_rad_total += 1
    else:
        if not df_match_ids[df_match_ids['match_id'] == row['matchId']]['didRadiantWin'].values[0]:
            fp_dire_wins += 1
        fp_dire_total += 1
fp_rad_winrate = fp_rad_wins / fp_rad_total
fp_dire_winrate = fp_dire_wins / fp_dire_total
fp_total_winrate = (fp_rad_wins + fp_dire_wins) / (fp_rad_total + fp_dire_total)
rad_total_winrate = len(df_match_ids[df_match_ids['didRadiantWin'] == True]) / len(df_match_ids)
dire_total_winrate = len(df_match_ids[df_match_ids['didRadiantWin'] == False]) / len(df_match_ids)
print(f'Radiant first-pick win rate: {fp_rad_winrate}\
    \nDire first-pick win rate: {fp_dire_winrate}\
    \nTotal first-pick win rate: {fp_total_winrate}')
print(f'Radiant win rate overall: {rad_total_winrate}\
      \nDire win rate overall: {dire_total_winrate}')


Radiant first-pick win rate: 0.5677966101694916    
Dire first-pick win rate: 0.4706208425720621    
Total first-pick win rate: 0.5090482573726541
Radiant win rate overall: 0.544      
Dire win rate overall: 0.456
